# ADAN Quant ML Research: Phase 1

This notebook focuses on analyzing the features collected by ADAN's Node.js engine natively in Python, connecting directly to the `adan_data.db` SQLite database.

**Objectives:**
1. Load historical feature logs from SQLite into a Pandas DataFrame.
2. Perform Exploratory Data Analysis (EDA) on the features.
3. Train a basic XGBoost model to evaluate feature importance and calculate Edge/Brier score.

In [2]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, classification_report
from xgboost import XGBClassifier

# Prettier plots
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (10, 6)

Task was destroyed but it is pending!
task: <Task pending name='Task-121' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/benjaminfuentes/Desktop/adan-pred/quant/venv/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-122' coro=<Kernel.shell_main() running at /Users/benjaminfuentes/Desktop/adan-pred/quant/venv/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/benjaminfuentes/Desktop/adan-pred/quant/venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/benjaminfuentes/Desktop/adan-pred/quant/venv/lib/python3.13/site-packages/xgboost/__init__.py:18: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  from .training import cv, train
Task was destroyed but it is pending!
task: <Task pending name='Task-122' coro=<Kernel.shell_main() running at /Users/benjaminfuentes/Desktop/adan-pred/quant/venv/lib/python3

In [2]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, classification_report
from xgboost import XGBClassifier

# Prettier plots
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (10, 6)

Task was destroyed but it is pending!
task: <Task pending name='Task-121' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/benjaminfuentes/Desktop/adan-pred/quant/venv/lib/python3.13/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-122' coro=<Kernel.shell_main() running at /Users/benjaminfuentes/Desktop/adan-pred/quant/venv/lib/python3.13/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/benjaminfuentes/Desktop/adan-pred/quant/venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/benjaminfuentes/Desktop/adan-pred/quant/venv/lib/python3.13/site-packages/xgboost/__init__.py:18: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  from .training import cv, train
Task was destroyed but it is pending!
task: <Task pending name='Task-122' coro=<Kernel.shell_main() running at /Users/benjaminfuentes/Desktop/adan-pred/quant/venv/lib/python3

### 1. Load Data from SQLite

In [10]:
# Connect to the local SQLite DB generated by sync_to_sqlite.py
conn = sqlite3.connect('adan_data.db')

# Group by and load all features
query = """
SELECT * FROM features 
WHERE resolved = 1 AND won IS NOT NULL
"""
df = pd.read_sql(query, conn)
conn.close()

print(f"Loaded {len(df)} resolved trades from the database.")
df.head()

Loaded 0 resolved trades from the database.


,trade_id,timestamp,asset,fear_greed,funding_rate,trend_strength,vol_ratio,utc_hour,edge,llm_confidence,resolved,won


### 2. Feature Engineering & Selection

In [5]:
# Drop rows with missing crucial features
df_clean = df.dropna(subset=['fear_greed', 'funding_rate', 'trend_strength', 'vol_ratio', 'won'])

# Select numerical features for the model
features = ['fear_greed', 'funding_rate', 'trend_strength', 'vol_ratio', 'utc_hour', 'edge', 'llm_confidence']
target = 'won'

X = df_clean[features]
y = df_clean[target].astype(int)

# Train/Test split (80% train, 20% test)
# Note: In real quant pipelines, we use Walk-Forward Cross Validation (Time Series Split) to avoid look-ahead bias.
# We will implement PurgedKFold in future iterations.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

print(f"Training set: {len(X_train)} samples")
print(f"Testing set:  {len(X_test)} samples")

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

### 3. Model Training (XGBoost)

In [6]:
# Initialize XGBoost Classifier
# We use binary logistic regression under the hood for probabilities
model = XGBClassifier(
    n_estimators=100,
    max_depth=3, 
    learning_rate=0.05,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

# Train model
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("Model training complete.")

NameError: name 'X_train' is not defined

### 4. Evaluation & Brier Score

In [7]:
# Predict probabilities (the true "Confidence/Edge")
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Predict binary classes (Threshold 0.5)

y_pred = model.predict(X_test)

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
brier = brier_score_loss(y_test, y_pred_proba)

print(f"Accuracy on Test Set: {acc:.2%}")
print(f"Brier Score: {brier:.4f} (Lower is better, < 0.20 is excellent)")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

NameError: name 'X_test' is not defined

### 5. Feature Importance

In [8]:
# Plot Feature Importances
importances = model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 6))
plt.title('Feature Importances (XGBoost)')
plt.barh(range(len(indices)), importances[indices], color='b', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

NotFittedError: need to call fit or load_model beforehand